In [60]:
import os
import faiss
from dotenv import load_dotenv
from transformers import AutoTokenizer

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
    Settings,
    set_global_tokenizer,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.vector_stores.faiss import FaissVectorStore

In [61]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [62]:
set_global_tokenizer(AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5").encode)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.chunk_size = 450
Settings.chunk_overlap = 50

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [63]:
documents = SimpleDirectoryReader(
    input_dir="/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data",
    exclude=[
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/README.md",
        "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/paul_graham_essay.txt",
    ],
).load_data(num_workers=4)

In [64]:
print(f"Loaded {len(documents)} documents.")

Loaded 11 documents.


In [65]:
documents[0].metadata

{'file_path': '/Users/mohitag/Documents/Projects/RAG_SQL_Project/Data/10min.rst',
 'file_name': '10min.rst',
 'file_type': 'text/x-rst',
 'file_size': 18933,
 'creation_date': '2026-07-21',
 'last_modified_date': '2026-07-21'}

In [66]:
d = 384
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [67]:
# from local drives
vector_store_local = FaissVectorStore.from_persist_dir(
    "/Users/mohitag/Documents/Projects/RAG_SQL_Project/Storage"
)
storage_context_local = StorageContext.from_defaults(
    persist_dir="/Users/mohitag/Documents/Projects/RAG_SQL_Project/Storage",
    vector_store=vector_store_local,
)
index_local = load_index_from_storage(storage_context_local)

In [68]:
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, show_progress=True
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/717 [00:00<?, ?it/s]

In [69]:
print(len(index.docstore.docs))

717


In [70]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print("--- CHUNK ---")
    print("Source file:", node.metadata.get("file_name"))
    print("Text preview:", node.text[:300])
    print()

--- CHUNK ---
Source file: 10min.rst
Text preview: .. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy 

--- CHUNK ---
Source file: 10min.rst
Text preview: Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a :class:`DataFrame` by passing a NumPy array with a datetime index using :func:`date_range`
and labeled columns

--- CHUNK ---
Source file: 10min.rst
Text preview: Here's a subset of the attributes that
will be completed:

.. ipython::

   @verbatim
   In [1]: df2.<TAB>  # noqa: E225, E999
   df2.A                  df2.bool
   df2.abs                df2.boxplot
   df2.add                df2.C
   df2.add_

In [71]:
for node_id, node in list(index.docstore.docs.items())[:5]:
    print(len(node.text), "characters")

918 characters
1200 characters
804 characters
879 characters
1004 characters


In [72]:
for node_id, node in list(index.docstore.docs.items())[:12]:
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
    print(len(tok.encode(node.text)), "tokens")

297 tokens
398 tokens
263 tokens
285 tokens
325 tokens
302 tokens
259 tokens
242 tokens
408 tokens
270 tokens
352 tokens
384 tokens


In [73]:
print(index.ref_doc_info)

{'e64d977f-b612-4419-a59a-6da00ab0859b': RefDocInfo(node_ids=['1c6e73dc-5c30-4b49-bf16-efc7834cde7a', '9703c979-2627-4975-b728-a5888e13bf29', 'ffbb39ad-3f01-4d22-b604-c21535da3017', '18d7d2d8-770b-4b62-8ed6-389abd62446b', '72cbf112-9ad6-46a5-abb4-90813c7f5643', '4261da8c-8a1c-4c42-83ea-42ba6dc1880b', '15e1d31c-ffe5-42a7-afff-41e71f4cb03c', 'dfb0326f-741f-4569-97cc-b5a117ca1db8', '05a866fe-c8c3-4c30-ba61-8e9793dc669c', '8416d2b1-a633-4e1d-a32a-687c83d47860', 'e6276955-43ee-440d-95c4-5338c11ef504', '19e4c4ac-95de-4ab4-a8a2-d70d9daa0ca3', '3b2f45ba-85a2-490d-a64f-e7b7a2983a0e', '28b2f462-e801-4b6b-bad1-e41c1a1d82eb', '99d55012-653f-4c24-a107-bd2db7964b7f', '68746aab-1020-4105-af3f-5488c512b544', '314b8a20-5de5-49eb-ae40-1244b09d486a', '508609c8-fa13-4cbe-98ae-17fa3145dc49', '3b1c7e7f-d581-44b6-86c9-64c6a64911ca', '5e02d521-2cae-449b-bb83-60e93388f9d4', 'edf8dd75-dfbb-4cf7-8410-57dfa7ff20c0', '7f6b6941-22f4-443a-8aa7-6475b63d2f6a'], metadata={'file_path': '/Users/mohitag/Documents/Projects

In [74]:
print(documents[0].text)

.. _10min:

{{ header }}

********************
10 minutes to pandas
********************

This is a short introduction to pandas, geared mainly for new users.
You can see more complex recipes in the :ref:`Cookbook<cookbook>`.

Customarily, we import as follows:

.. ipython:: python

   import numpy as np
   import pandas as pd

Basic data structures in pandas
-------------------------------

pandas provides two types of classes for handling data:

1. :class:`Series`: a one-dimensional labeled array holding data of any type
    such as integers, strings, Python objects etc.
2. :class:`DataFrame`: a two-dimensional data structure that holds data like
   a two-dimension array or a table with rows and columns.

Object creation
---------------

See the :ref:`Intro to data structures section <dsintro>`.

Creating a :class:`Series` by passing a list of values, letting pandas create
a default :class:`RangeIndex`.

.. ipython:: python

   s = pd.Series([1, 3, 5, np.nan, 6, 8])
   s

Creating a 

In [75]:
Settings.llm = HuggingFaceInferenceAPI(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    token=hf_token,
    temperature=0.2,
    max_tokens=256,
    provider="auto",
)

In [76]:
query_engine = index.as_query_engine()

In [77]:
response = query_engine.query("how to create a csv file?")

In [78]:
print(response.response)

You can use the `to_csv` method of a pandas DataFrame or Series to create a CSV file. This method takes several arguments, but only the first one is required. The required argument is the file path where the CSV file will be stored.


In [79]:
response = query_engine.query("create a dataframe?")

In [80]:
print(response.response)

To create a DataFrame, you can use the `pd.DataFrame()` function, passing in a dictionary or a list of lists, where each inner list represents a row in the DataFrame. For example:

```python
import pandas as pd

data = {'Name': ['John', 'Anna', 'Peter', 'Linda'],
        'Age': [28, 24, 35, 32],
        'Country': ['USA', 'UK', 'Australia', 'Germany']}

df = pd.DataFrame(data)
print(df)
```

This will create a DataFrame with three columns: `Name`, `Age`, and `Country`, and four rows of data. The output will look something like this:

```
     Name  Age    Country
0    John   28        USA
1    Anna   24         UK
2   Peter   35  Australia
3   Linda   32    Germany
```


In [81]:
query_engine_local = index_local.as_query_engine()


In [82]:
print(query_engine_local.query("load dataframe from cloud with examples").response)

Loading a DataFrame from the cloud can be achieved through various methods, depending on the cloud storage system you're using. Here are a few examples:

1. **AWS S3**: You can use the `boto3` library in Python to load a DataFrame from an S3 bucket. First, install `boto3` using pip: `pip install boto3`. Then, import the library and use the `S3` client to load the data.

   ```python
   import boto3
   s3 = boto3.client('s3')
   df = pd.read_csv('s3://your-bucket-name/your-file.csv')
   ```

2. **Google Cloud Storage**: You can use the `google-cloud-storage` library in Python to load a DataFrame from a Cloud Storage bucket. First, install the library using pip: `pip install google-cloud-storage`. Then, import the library and use the `Client` to load the data.

   ```python
   from google.cloud import storage
   client = storage.Client()
   bucket = client.get_bucket('your-bucket-name')
   blob = bucket.get_blob('your-file.csv')
   df = pd.read_csv(blob.download_as_string())
   ```

3. *

In [83]:
print(query_engine_local.query("operations on dataframes").response)

Operations on DataFrames can be performed using various methods, including the `query` method, which allows for flexible and expressive querying of the data. This method supports a range of operators, including comparison operators, logical operators, and set operators, enabling users to craft complex queries that can be executed on the DataFrame.

Some common operations on DataFrames include filtering, sorting, grouping, and merging. Filtering involves selecting a subset of rows based on conditions, while sorting rearranges the rows in a specific order. Grouping allows for aggregation of data across different categories, and merging enables the combination of data from multiple DataFrames.

The `query` method provides a powerful way to perform these operations, allowing users to write queries that are both readable and efficient. By leveraging the expressive power of Python's syntax, users can craft queries that are easy to understand and maintain, making it easier to work with comple

In [94]:
faiss.serialize_index(index.storage_context.vector_store._faiss_index)

array([ 73, 120,  70, ..., 184,  34, 188], shape=(1101357,), dtype=uint8)